# YOLO Crab Dataset Generator — Google Colab
Generate a synthetic dataset of crab images for training a YOLO object detection model.

## 1. Install Dependencies

In [ ]:
!pip install -q opencv-python-headless pyyaml tqdm

## 2. Upload Crab Images

Upload the three crab images: `green_crab.jpg`, `rock_crab.jpg`, `jonah_crab.png`.

In [ ]:
import os
from google.colab import files

os.makedirs("crabs", exist_ok=True)
uploaded = files.upload()

for fname, data in uploaded.items():
    dest = os.path.join("crabs", fname)
    with open(dest, "wb") as f:
        f.write(data)
    print(f"Saved: {dest}")

print("Files in crabs/:", os.listdir("crabs"))

## 3. Configure Dataset Parameters

In [ ]:
import cv2
import numpy as np
import random
import yaml
from tqdm import tqdm

BASE_DIR = "/content"

# --- Config ---
IMG_SIZE      = 640
N_TRAIN       = 400
N_VAL         = 100
CRABS_PER_IMG = (3, 8)  # min, max crabs per image

# Solo la specie invasiva viene etichettata.
# Rock e Jonah appaiono nelle immagini come distrattori negativi.
classes = {
    "green": 0,
}

CRAB_FILES = {
    "green": "crabs/green_crab.jpg",
    "rock":  "crabs/rock_crab.jpg",
    "jonah": "crabs/jonah_crab.png",
}

# --- Load crab images ---
crabs = {}
for name, rel_path in CRAB_FILES.items():
    img = cv2.imread(os.path.join(BASE_DIR, rel_path), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(f"Could not load crab image: {rel_path}")
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGRA)
    elif img.shape[2] == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2BGRA)
    crabs[name] = img
    print(f"Loaded {name}: {img.shape}")

# --- Create output dirs ---
for split in ("train", "val"):
    os.makedirs(os.path.join(BASE_DIR, "dataset", "images", split), exist_ok=True)
    os.makedirs(os.path.join(BASE_DIR, "dataset", "labels", split), exist_ok=True)

# --- Write data.yaml ---
yaml_path = os.path.join(BASE_DIR, "dataset", "data.yaml")
data_yaml = {
    "path":  os.path.join(BASE_DIR, "dataset"),
    "train": "images/train",
    "val":   "images/val",
    "nc":    len(classes),
    "names": [k for k, _ in sorted(classes.items(), key=lambda x: x[1])],
}
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)
print(f"\nSaved data.yaml -> {yaml_path}")
print(f"Classi etichettate: {data_yaml['names']} (nc={data_yaml['nc']})")
print("Rock e Jonah crab: presenti come distrattori, senza etichetta.")

## 4. Background Generation

In [ ]:
def random_background(size):
    """Generate a random ocean-toned or sandy background."""
    choice = random.randint(0, 2)
    bg = np.zeros((size, size, 3), dtype=np.uint8)
    if choice == 0:
        # Ocean color with noise
        base = np.array([random.randint(50, 130), random.randint(80, 160), random.randint(30, 80)], dtype=np.uint8)
        noise = np.random.randint(-20, 20, (size, size, 3), dtype=np.int16)
        bg = np.clip(base + noise, 0, 255).astype(np.uint8)
    elif choice == 1:
        # Sandy bottom
        base = np.array([random.randint(100, 180), random.randint(130, 200), random.randint(150, 220)], dtype=np.uint8)
        noise = np.random.randint(-30, 30, (size, size, 3), dtype=np.int16)
        bg = np.clip(base + noise, 0, 255).astype(np.uint8)
    else:
        # Vertical gradient
        top = np.array([random.randint(60, 120), random.randint(90, 160), random.randint(40, 100)])
        bot = np.array([random.randint(100, 200), random.randint(140, 210), random.randint(80, 150)])
        for row in range(size):
            t = row / size
            bg[row] = np.clip((1 - t) * top + t * bot, 0, 255).astype(np.uint8)
    return bg

print("random_background() defined.")

## 5. Crab Augmentation & Alpha Blending

In [ ]:
def paste_crab(bg, crab_bgra, x, y):
    """Alpha-blend a BGRA crab onto a BGR background at position (x, y)."""
    h, w = crab_bgra.shape[:2]
    alpha    = crab_bgra[:, :, 3:4].astype(np.float32) / 255.0
    crab_rgb = crab_bgra[:, :, :3].astype(np.float32)
    roi      = bg[y:y+h, x:x+w].astype(np.float32)
    blended  = alpha * crab_rgb + (1 - alpha) * roi
    bg[y:y+h, x:x+w] = blended.astype(np.uint8)


def augment_crab(crab):
    """Apply random scale, flip, rotation, and brightness to a BGRA crab."""
    scale = random.uniform(0.2, 0.6)
    w = max(1, int(crab.shape[1] * scale))
    h = max(1, int(crab.shape[0] * scale))
    crab = cv2.resize(crab, (w, h))

    # Random horizontal flip
    if random.random() < 0.5:
        crab = cv2.flip(crab, 1)

    # Random rotation
    angle = random.randint(0, 359)
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1)
    crab = cv2.warpAffine(crab, M, (w, h))

    # Random brightness (BGR channels only)
    factor = random.uniform(0.6, 1.4)
    crab = crab.copy()
    crab[:, :, :3] = np.clip(crab[:, :, :3].astype(np.float32) * factor, 0, 255).astype(np.uint8)

    return crab

print("paste_crab() and augment_crab() defined.")

## 6. Image & Label Generation

In [ ]:
def generate_image(idx, split):
    bg = random_background(IMG_SIZE)
    label_lines = []

    n = random.randint(*CRABS_PER_IMG)
    for _ in range(n):
        name = random.choice(list(crabs.keys()))
        crab = augment_crab(crabs[name])

        h, w = crab.shape[:2]
        if w >= IMG_SIZE or h >= IMG_SIZE:
            continue

        x = random.randint(0, IMG_SIZE - w)
        y = random.randint(0, IMG_SIZE - h)

        paste_crab(bg, crab, x, y)

        # Solo i green crab vengono etichettati; rock e jonah sono distrattori negativi
        if name in classes:
            xc = (x + w / 2) / IMG_SIZE
            yc = (y + h / 2) / IMG_SIZE
            ww = w / IMG_SIZE
            hh = h / IMG_SIZE
            label_lines.append(f"{classes[name]} {xc:.6f} {yc:.6f} {ww:.6f} {hh:.6f}")

    img_path   = os.path.join(BASE_DIR, "dataset", "images", split, f"{idx}.jpg")
    label_path = os.path.join(BASE_DIR, "dataset", "labels", split, f"{idx}.txt")
    cv2.imwrite(img_path, bg)
    with open(label_path, "w") as f:
        f.write("\n".join(label_lines))

print("generate_image() defined.")

## 7. Generate Dataset

In [ ]:
import shutil

for i in tqdm(range(N_TRAIN), desc="Train"):
    generate_image(i, "train")

for i in tqdm(range(N_VAL), desc="Val"):
    generate_image(i, "val")

print("Dataset generation complete!")

# Zip the dataset for easy download
zip_path = os.path.join(BASE_DIR, "dataset")
shutil.make_archive(zip_path, "zip", BASE_DIR, "dataset")
print(f"Dataset zipped: {zip_path}.zip")

## 8. Download Dataset

In [ ]:
from google.colab import files
files.download(os.path.join(BASE_DIR, "dataset.zip"))

## 9. Preview Generated Images

Display a 3×3 grid of training images with bounding boxes overlaid.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

CLASS_COLORS = {"green": (0, 1, 0), "rock": (1, 0.5, 0), "jonah": (0, 0.5, 1)}
id2name = {v: k for k, v in classes.items()}

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
axes = axes.flatten()

train_img_dir   = os.path.join(BASE_DIR, "dataset", "images", "train")
train_label_dir = os.path.join(BASE_DIR, "dataset", "labels", "train")

for ax, idx in zip(axes, random.sample(range(N_TRAIN), 9)):
    img_path   = os.path.join(train_img_dir,   f"{idx}.jpg")
    label_path = os.path.join(train_label_dir, f"{idx}.txt")

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"img {idx}", fontsize=8)

    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cls_id, xc, yc, ww, hh = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            x = (xc - ww / 2) * IMG_SIZE
            y = (yc - hh / 2) * IMG_SIZE
            w = ww * IMG_SIZE
            h = hh * IMG_SIZE
            color = CLASS_COLORS.get(id2name[cls_id], (1, 1, 1))
            rect = patches.Rectangle((x, y), w, h, linewidth=1.5, edgecolor=color, facecolor="none")
            ax.add_patch(rect)
            ax.text(x, y - 2, id2name[cls_id], color=color, fontsize=6, weight="bold")

plt.tight_layout()
plt.show()